In [ ]:
!pip install evaluate

# Negation Set Generation

In [ ]:
# Use LLM to generate negated hypothesis
import pandas as pd
import os
import ast
from datasets import load_dataset
import numpy as np
from tqdm.notebook import tqdm
from openai import OpenAI
from dotenv import load_dotenv


#NVIDIA appears to be offering generous free tokens.
load_dotenv()
openai_client=OpenAI(base_url="https://integrate.api.nvidia.com/v1",api_key=os.getenv('NVIDIA_API_KEY'))
ds=load_dataset("stanfordnlp/snli")
valset=ds['validation']

def generate_negated_hypothesis(client,dataset, batch_size=20,fname="/content/drive/MyDrive/negated_validation_set.xlsx"):
  premises=dataset['premise']
  hypotheses=dataset['hypothesis']

  if os.access(fname, os.F_OK):
    existing_df=pd.read_excel(fname)
  else:
    existing_df=pd.DataFrame({"premises":premises,"hypothesis":hypotheses,"negated_hypothesis":np.nan})
  negated_hypothesis=existing_df['negated_hypothesis']

  # Feed the model batches of premises and hypotheses
  for i in tqdm(range(0,len(valset),batch_size),desc="Progress"):
    if not pd.isna(existing_df.loc[i,'negated_hypothesis']):
      print(f"batch {i//batch_size} is not none; skipping")
      continue
    batch=valset[i:i+batch_size]
    premises=batch['premise']
    hypotheses=batch['hypothesis']
    content=""
    for p in range(len(premises)):
      content+=f"Pair {p}: Premise: {premises[p]}; Hypothesis: {hypotheses[p]}\n"
    #response=client.models.generate_content(model=model,contents=prompt)
    prompt=f"negate the following hypotheses by only changing the verb. return the result in a python list. You must keep the original order. return only the list, nothing else, not even quotes and back ticks.\n {content}"
    #response=call_api(groq_client,prompt)
    completion=client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role":"user","content":prompt}
        ],
        temperature=0.2,
        top_p=1,
        max_tokens=6000,
        stream=False
    )
    response=completion.choices[0].message.content
    try:
      new_nh=ast.literal_eval(response)
    except Exception as e:
      print(f"Exception: {e}")
      continue
    try:
      negated_hypothesis[i:i+batch_size]=new_nh
    except Exception as e:
      print(f"Exception: {e}")
      print(f"Length of the new hypotheses: {len(new_nh)}")
      continue
    existing_df['negated_hypothesis']=negated_hypothesis
    existing_df['label']=valset['label']
    existing_df.to_excel(fname,index=False)
generate_negated_hypothesis(openai_client,valset)

In [ ]:
# Put the model to the working directory
!unzip /content/drive/MyDrive/Qwen3-0.6B_finetuned.zip -d .

In [38]:
import pandas as pd
import os
import ast
from datasets import load_dataset, Dataset
import numpy as np
from tqdm.notebook import tqdm
from openai import OpenAI
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer
# Load the finetuned model
model_path='/content/Qwen3-0.6B_finetuned/final_model'
tokenizer=AutoTokenizer.from_pretrained(model_path)
model=AutoModelForCausalLM.from_pretrained(model_path,device_map='auto',torch_dtype='auto')

# Load the base model
base_model_name="Qwen/Qwen3-0.6B"
base_model=AutoModelForCausalLM.from_pretrained(base_model_name,device_map="auto",torch_dtype="auto")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

# Evaluation On Multi-NLI dataset

In [41]:
import re
import evaluate
import torch
def extract_label(text):
  pattern=re.compile(r'(?i)Label:\s*(\w+)')
  index_mapping={'entailment':0,'neutral':1,'contradiction':2,}
  matched=pattern.search(text)
  if matched:
    label=matched.group(1).lower()
    return index_mapping.get(label,-1)
  return -1
@torch.no_grad()
def evaluate_model(model,dataset,gt='label',batch_size=2):
  metric=evaluate.load('accuracy')
  predictions=[]
  gt=dataset[gt]
  for i in tqdm(range(0,len(dataset),batch_size),desc="Progress"):
    samples=dataset[i:i+batch_size]
    inputs=tokenizer(samples['prompt'],return_tensors="pt",padding=True, padding_side="left",truncation=True).to(model.device)
    response=model.generate(**inputs,max_new_tokens=10)
    strings=tokenizer.batch_decode(response,skip_special_tokens=True)
    labels=[extract_label(r) for r in strings]
    predictions.extend(labels)
  result=metric.compute(predictions=predictions,references=gt[0:len(predictions)])
  return result

def preprocessing(sample):
  sample['prompt']=f"Premis: {sample['premise']}. Hypothesis: {sample['hypothesis']} Label:"
  return sample



In [43]:
multi_nli=load_dataset("nyu-mll/multi_nli")
val_mismatched=multi_nli['validation_mismatched']

# Adjust number of samples to be evaluated below.
num_samples=100
val_mismatched=val_mismatched.map(preprocessing).shuffle().select(range(num_samples))


base_model_result=evaluate_model(base_model,val_mismatched)
finetuned_model_result=evaluate_model(model,val_mismatched)

print(f"Base Model Accuracy: {base_model_result['accuracy']}; Finetuned Model Accuracy: {finetuned_model_result['accuracy']}")

Progress:   0%|          | 0/50 [00:00<?, ?it/s]

Progress:   0%|          | 0/50 [00:00<?, ?it/s]

Base Model Accuracy: 0.0; Finetuned Model Accuracy: 0.75


# Evaluation on negated dataset

In [44]:
def negation_preprocess(sample):
  if sample['label']==0:
    return 2
  else:
    return 0

def negation_prompt(sample):
  sample['prompt']=f"Premis: {sample['premises']}. Hypothesis: {sample['negated_hypothesis']} Label:"
  return sample


In [45]:
path="/content/drive/MyDrive/negated_validation_set.xlsx"
df=pd.read_excel(path)
mask=df['label']!=1
df=df[mask]
df['negated_label']=df[mask].apply(negation_preprocess,axis=1)
df=df.apply(negation_prompt,axis=1)
samples=100
negated_dataset=Dataset.from_pandas(df).select(range(samples))
base=evaluate_model(base_model,negated_dataset,gt='negated_label')
finetuned=evaluate_model(model,negated_dataset,gt='negated_label')
print(f"Base model accuracy: {base['accuracy']}; finetuned model accuracy: {finetuned['accuracy']}")

/tmp/ipykernel_1566/4212806605.py:5: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df['negated_label']=df[mask].apply(negation_preprocess,axis=1)


Progress:   0%|          | 0/50 [00:00<?, ?it/s]

Progress:   0%|          | 0/50 [00:00<?, ?it/s]

Base model accuracy: 0.0; finetuned model accuracy: 0.46
